In [11]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from gameboard import gameboard

In [12]:
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()
# if GPU is to be used
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
print(device)

cuda


In [13]:
Transition= namedtuple("Transition",("State","action","reward","next_state"))

class ReplayMemory:
    def __init__(self,capacity):
        self.memory = deque([],maxlen=capacity)

    def add_memory(self,*args):
        self.memory.append(Transition(*args))
    def sample_memory(self,batch_size):
        return random.sample(self.memory,batch_size)
    def __len__(self):
        return len(self.memory)

In [14]:
class DQN(nn.Module):
    def __init__(self,n_observations,n_actions,hidden_size):
        super().__init__()
        self.layer1= nn.Linear(n_observations,hidden_size)
        self.layer2 =nn.Linear(hidden_size,hidden_size)
        self.layer3= nn.Linear(hidden_size,n_actions)
    def forward(self,observations):
        x = F.relu(self.layer1(observations))
        x = F.relu(self.layer2(x))
        return self.layer3(x)





In [15]:

BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 2500
TAU = 0.005
LR = 3e-4
hidden_size=128
import pygame  

pygame.init()
clock = pygame.time.Clock()
running = True



screen = pygame.display.set_mode((640,640))


action_dict={
    0:[1,0,0,0],
    1:[0,1,0,0],
    2:[0,0,1,0],
    3:[0,0,0,1],
    4:[1,0,1,0],
    5:[1,0,0,1],
    6:[0,1,1,0],
    7:[0,1,0,1]
}
n_actions = len(action_dict)
game = gameboard(screen,20,800)
def game_step(action):
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            return None, 0
    game.dt = 1/60
    observation,reward = game.updategame(action=action)
    observation_list=[]

    if observation:
        for key,value in observation.items():
            observation_list+=list(value)

    pygame.display.flip()
    return observation_list,round(reward)



In [16]:
action =0
observation,reward = game_step(action_dict[action])
n_observations = len(observation)
policy_net = DQN(n_observations, n_actions,hidden_size).to(device)
target_net = DQN(n_observations, n_actions,hidden_size).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)


steps_done = 0

In [17]:
def choose_action(observation):
    global steps_done
    global action
    with torch.no_grad():
        exp = EPS_START*math.exp(-steps_done/EPS_DECAY)+EPS_END
        steps_done+=1
        policy_net.to(device)
        if random.random() < exp:
            if steps_done%10 ==0:
                action = random.choice(range(n_actions))
            return action
        else:
            qscores = policy_net(torch.tensor(observation,dtype=torch.float32,device=device))
            _,index=qscores.max(0)
            return index.item()

In [18]:
memstuff=[]
frame_count=0
def frame_step():
    global observation
    global frame_count 
    action = choose_action(observation)
    next_state,reward = game_step(action_dict[action])
    if next_state and game.sneks:
            memory.add_memory(torch.tensor(observation),torch.tensor([action]),torch.tensor(reward),torch.tensor(next_state) if next_state else next_state)
            observation = next_state 
            if reward == 0:
                frame_count+=1
                if frame_count > 700:
                    return False
            else: frame_count=0
            return True
    else: return False

    


frame_step()
criterion = nn.SmoothL1Loss()
def optimizer_step():
    if len(memory) < BATCH_SIZE:
        return
    else:
        optimizer.zero_grad()
        transitions = memory.sample_memory(BATCH_SIZE)
        batch = Transition(*zip(*transitions))
        state_batch = torch.stack(batch.State).to(device)
        non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                            batch.next_state)), device=device, dtype=torch.bool)
        non_final_next_states = torch.stack([s for s in batch.next_state
                                                    if s is not None]).to(device)
        action_batch = torch.stack(batch.action).to(device)
        reward_batch = torch.stack(batch.reward).to(device)
        next_state_values = torch.zeros(BATCH_SIZE,device=device)
        with torch.no_grad():
            next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
        target=reward_batch + GAMMA*next_state_values
        qvalue=policy_net(state_batch).gather(1,action_batch).squeeze(1)
        loss =criterion(target,qvalue)
        loss.backward()
        torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
        optimizer.step()
        policy_state_dict = policy_net.state_dict()
        target_state_dict = target_net.state_dict()
        for key in policy_state_dict:
            target_state_dict[key]= (1-TAU)*target_state_dict[key]+TAU*policy_state_dict[key]
        target_net.load_state_dict(target_state_dict)




In [19]:
episode_reward=[]
def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_reward, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.plot(durations_t.numpy())
    if len(episode_reward)>10:
        means = durations_t.unfold(0,10,1).mean(1).view(-1)
        means= torch.cat((torch.zeros(9),means))
        plt.plot(means.numpy())
    plt.pause(0.001)
    plt.show()
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

In [20]:
n_episodes = 600
for i in range(n_episodes):
    frame_count=0
    game = gameboard(screen,20,800)
    while True:
        done = frame_step()
        optimizer_step()
        if not done:
            episode_reward.append(game.agent_point)
            plot_durations()
            break
print('Complete')
plot_durations(show_result=True)
plt.ioff()
plt.show()
        

error: font not initialized

<Figure size 640x480 with 0 Axes>

In [ ]:
pygame.quit()
     

Batch_size=128
learning rate =3e-4
noeps=200
hidden size=128 
gamma =0.99 

### Observations
- Average points per game quickly rose but plateued 
- Large number of games with minimum ammount of points (~10)
- Few large outliers of points (1000-2000) observed attributed to "getting lucky" and having other snakes(force based ai snakes) kill each other 
- learned preference for bottom left of playable area attributed to top and bottom borders overlapping with food leading to frequent death so bottom and left borders which do not overlap with food is prefered and predictibly safer 
- Head hunting other snakes behavior appears to mimic force based ai snakes 
- Has not reliably learned hunting other snakes improvement might be made by attributing a reward to incentivize agent
- Learned to avoid other snakes bodies quickly but not as reliably as episodes went on
- Learned eating food but doesnt reliably follow food clusters 
- average reward Plateus at around 100 - 200 points attributed to large outliers and also large frequency of quick failures
- Avoidance of bottom border learned fairly well 


In [22]:
torch.save(policy_net.state_dict(), "policy_net.pth")

In [ ]:
torch.save({
    "policy_net": policy_net.state_dict(),
    "target_net": target_net.state_dict(),
    "optimizer": optimizer.state_dict(),
    "n_observations": n_observations,
    "hidden_size":hidden_size
}, "dqn_checkpoint_1.pth")

: 